# N31 — Strategy Orchestrator

End-to-end multi-agent supervisor. Integrates N25–N30 sub-agents through three
processing layers: dynamic MoE-style routing, Monte Carlo simulation over the
probabilistic outputs, and structured LLM synthesis of the final recommendation.

**Architecture**
- Layer 1: dynamic routing — decides which sub-agents to call each lap
- Layer 2: Monte Carlo simulation — samples from each sub-agent's uncertainty outputs
- Layer 3: LLM synthesis — aggregates all reasoning + MC scores into `StrategyRecommendation`

**References**: Heilmeier et al. (2020) ApplSci 10/4229 (MC motorsport sim),
Wang et al. (2024) MoA arXiv:2406.04692 (reasoning aggregation pattern),
Liu et al. (2024) DeLLMa arXiv:2402.02392 (decision under uncertainty with LLM).


---

## Step 0 — Setup & model loading

Imports, repo root, LLM client, FastF1 cache. `OrchestratorCFG` holds the
orchestrator's runtime constants: MC simulation parameters, MoE routing
thresholds, and the LLM client used in Layer 3 synthesis.

| Constant | Value | Purpose |
|---|---|---|
| `n_sim` | 500 | MC draws per strategy candidate |
| `sc_prob_threshold` | 0.30 | activates N30 if N27.sc_prob_3lap exceeds this |
| `risk_tolerance_default` | 0.5 | α in `score = α·E[S] + (1−α)·P10[S]` |


In [27]:
import json, sys
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import pandas as pd
import fastf1
from pydantic import BaseModel, Field
from typing import Optional
from types import SimpleNamespace
from langchain_openai import ChatOpenAI
repo_root = Path.cwd()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [28]:
@dataclass
class OrchestratorCFG:
    """Runtime configuration for the Strategy Orchestrator (N31).

    n_sim controls Monte Carlo draws per strategy candidate in Layer 2. 500 draws
    keep variance of the mean below 0.01 position units within lap-level latency.

    sc_prob_threshold is the N27.sc_prob_3lap cutoff above which N30 is activated
    to retrieve safety-car regulation context for the pit decision.

    risk_tolerance_default (α) weights expected value vs worst-case in the MC
    score: score(S) = α·E[S] + (1−α)·P10[S]. α=1.0 aggressive, α=0.0 conservative.

    temperature=0.0 ensures deterministic structured output from Layer 3 LLM.
    """

    model_name: str = "local-model"
    base_url: str = "http://localhost:1234/v1"
    temperature: float = 0.0
    n_sim: int = 500
    sc_prob_threshold: float = 0.30
    risk_tolerance_default: float = 0.5

    def __post_init__(self):
        root = Path.cwd()
        while not (root / ".git").exists():
            root = root.parent

        fastf1.Cache.enable_cache(str(root / "data" / "cache" / "fastf1"))

        self.export_dir = root / "data" / "models" / "agents"
        self.export_dir.mkdir(parents=True, exist_ok=True)

        self.llm = ChatOpenAI(
            model=self.model_name,
            base_url=self.base_url,
            api_key="lm-studio",
            temperature=self.temperature,
        )

In [29]:
CFG = OrchestratorCFG()

print(f"LLM          : {CFG.model_name} @ {CFG.base_url}")
print(f"N_SIM        : {CFG.n_sim}")
print(f"SC threshold : {CFG.sc_prob_threshold}")
print(f"α default    : {CFG.risk_tolerance_default}")
print(f"EXPORT_DIR   : {CFG.export_dir}")

LLM          : local-model @ http://localhost:1234/v1
N_SIM        : 500
SC threshold : 0.3
α default    : 0.5
EXPORT_DIR   : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\models\agents


---

## Step 1 — `RaceState` + MoE routing

`RaceState` is the per-lap context object passed to the orchestrator. It is a
Pydantic `BaseModel` (not a dataclass) so that Pydantic validates field types and
ranges before any agent is called — bad inputs fail loudly at the boundary.

`_decide_agents_to_call` implements Layer 1 routing. N25, N26, N27 and N29 are
always activated. N28 and N30 are conditional:

| Condition | Agent activated |
|---|---|
| N26.warning_level == "PIT_SOON" | N28 |
| N29.alerts contains PROBLEM or WARNING radio intent | N28 |
| N27.sc_prob_3lap > CFG.sc_prob_threshold | N30 |
| N28 activated | N30 (regulation constraint check) |

Returns a `set[str]` of agent keys so the caller can branch without inspecting
the outputs again.

---

### Routing design note — rule-based sparse activation, not a learned MoE

The term "MoE" here is borrowed from the mixture-of-experts paradigm
(Jacobs et al. 1991; Shazeer et al. 2017 — Outrageously Large Neural Networks)
but the mechanism is fundamentally different from a traditional MoE:

| | Traditional MoE | This system |
|---|---|---|
| Gating | Learned softmax network | Deterministic if-else rules |
| Expert weights | Continuous (soft routing) | Binary (activate / skip) |
| Training | End-to-end with gating loss | No gating training data |
| Output combination | Weighted sum of expert outputs | Sequential: activated agents feed MC simulation |

The closer academic reference is **conditional computation** (Bengio et al. 2013) —
selectively activating parts of a system based on input state — applied here at the
agent level. In the multi-agent literature this maps to the **selective tool invocation**
pattern used in LangGraph and AutoGen (Wu et al. 2023), where a controller decides
which tools/agents to call per step.

**Why deterministic routing is the right choice here:**

1. *Interpretability* — a strategy engineer must be able to understand why N28 was called.
   "Tyre warning = PIT_SOON triggered Pit Strategy agent" is auditable.
   A softmax gating vector is not.

2. *Reliability* — when `PIT_SOON` is true, calling N28 is mandatory, not probabilistic.
   Soft routing could assign N28 a weight of 0.6 and silently miss a pit window.

3. *No training signal* — a learned gating network needs labeled examples of
   (RaceState → optimal agent subset). No such dataset exists.

4. *Latency* — an LLM-as-router (e.g. ReAct-style planner deciding which agents
   to call) adds one extra round-trip before any real computation starts.
   For a real-time system making per-lap decisions every ~90 seconds, this matters.

**What could be made more complex (and why it's deferred):**

- *Adaptive thresholds*: instead of a fixed `sc_prob_threshold = 0.30`, learn
  per-circuit optimal thresholds from historical race data (logistic regression on
  retrospective pit decisions). Straightforward to add in v0.9, deferred to keep
  N31 focused on orchestration architecture.

- *Confidence-weighted prompt injection*: sub-agent outputs already carry a
  `reasoning` string. The LLM synthesis step (Step 3) could weight each agent's
  reasoning by its model's calibration confidence before injection. Implicit now;
  could be explicit with a weighting coefficient per agent.

- *LLM-as-router*: let the LLM itself decide which agents to invoke, given the
  current RaceState. Rejected for latency and non-determinism — a strategy call
  must be auditable and fast, not dependent on an LLM reasoning about whether
  to call another LLM.

In [30]:


class RaceState(BaseModel):
    """Per-lap context slice passed to the Strategy Orchestrator.

    driver identifies the driver whose strategy is being evaluated — all gap
    and pace features are relative to this driver.

    lap and total_laps enable race-percentage features used by N28 (lap_race_pct)
    and the MC simulation for fuel load estimation.

    compound and tyre_life are the current stint values forwarded to N26.

    gap_ahead_s and pace_delta_s are the primary inputs for N27 overtake scoring.

    weather fields (air_temp, track_temp, rainfall) are forwarded to N14 (SC model)
    as contextual features.

    radio_msgs and rcm_events are pre-filtered to the current lap ±1 window by the
    caller before passing to N29 — N31 does not do the filtering itself.

    risk_tolerance (α) is the MC score weight: score = α·E[S] + (1−α)·P10[S].
    Validated in [0, 1] by Pydantic; default 0.5 is neutral.
    """

    driver:         str
    lap:            int
    total_laps:     int
    position:       int
    compound:       str
    tyre_life:      int
    gap_ahead_s:    float
    pace_delta_s:   float
    air_temp:       float
    track_temp:     float
    rainfall:       bool = False
    radio_msgs:     list[dict] = Field(default_factory=list)
    rcm_events:     list[dict] = Field(default_factory=list)
    risk_tolerance: float = Field(default=0.5, ge=0.0, le=1.0)


def _decide_agents_to_call(
    tire_warning: str,
    sc_prob_3lap: float,
    radio_alerts: list[dict],
) -> set[str]:
    """Layer 1 MoE routing — returns set of conditional agent keys to activate.

    N25, N26, N27, N29 are always called by run_strategy_orchestrator and are
    not returned here. This function only decides N28 and N30.

    tire_warning is TireOutput.warning_level ("OK" | "MONITOR" | "PIT_SOON").
    sc_prob_3lap is RaceSituationOutput.sc_prob_3lap from N27.
    radio_alerts is RadioOutput.alerts — each dict has keys 'source' and 'intent'
    or 'event_type'.

    N28 is activated if the tyre is near the cliff or if radio signals a problem.
    N30 is activated whenever N28 is active (regulation check on the pit decision)
    or if SC probability is high (fetch SC procedure articles).
    """
    activate: set[str] = set()

    if tire_warning == "PIT_SOON":
        activate.add("N28")

    alert_intents = {a.get("intent", "") for a in radio_alerts}
    if alert_intents & {"PROBLEM", "WARNING"}:
        activate.add("N28")

    if sc_prob_3lap > CFG.sc_prob_threshold:
        activate.add("N30")

    if "N28" in activate:
        activate.add("N30")

    return activate


In [31]:
# Smoke test — routing logic
state = RaceState(
    driver="NOR", lap=18, total_laps=57, position=3,
    compound="C2", tyre_life=20, gap_ahead_s=1.2, pace_delta_s=-0.3,
    air_temp=32.0, track_temp=48.0, rainfall=False,
    radio_msgs=[], rcm_events=[], risk_tolerance=0.5,
)
print("RaceState OK:", state.driver, "lap", state.lap, "tyre_life", state.tyre_life)

# Case 1 — tyre cliff
a1 = _decide_agents_to_call("PIT_SOON", 0.10, [])
print("PIT_SOON, sc=0.10, no alerts  →", a1)  # {N28, N30}

# Case 2 — SC spike, no pit need
a2 = _decide_agents_to_call("MONITOR", 0.45, [])
print("MONITOR,  sc=0.45, no alerts  →", a2)  # {N30}

# Case 3 — radio PROBLEM
a3 = _decide_agents_to_call("OK", 0.10, [{"source": "radio", "intent": "PROBLEM"}])
print("OK,       sc=0.10, PROBLEM     →", a3)  # {N28, N30}

# Case 4 — all quiet
a4 = _decide_agents_to_call("MONITOR", 0.05, [])
print("MONITOR,  sc=0.05, no alerts  →", a4)  # set()


RaceState OK: NOR lap 18 tyre_life 20
PIT_SOON, sc=0.10, no alerts  → {'N28', 'N30'}
MONITOR,  sc=0.45, no alerts  → {'N30'}
OK,       sc=0.10, PROBLEM     → {'N28', 'N30'}
MONITOR,  sc=0.05, no alerts  → set()


### Step 1 results — `RaceState` + `_decide_agents_to_call`

`RaceState` instantiates and validates cleanly as a Pydantic `BaseModel`.

Routing smoke test — four cases:

| tire_warning | sc_prob_3lap | radio_alerts | agents activated |
|---|---|---|---|
| `PIT_SOON` | 0.10 | — | `{N28, N30}` |
| `MONITOR` | 0.45 | — | `{N30}` |
| `OK` | 0.10 | `PROBLEM` | `{N28, N30}` |
| `MONITOR` | 0.05 | — | `set()` |

N28 triggers whenever the tyre is near the cliff or a PROBLEM/WARNING alert
is present in the radio. N30 activates in cascade whenever N28 is active
(regulation constraint check before any pit recommendation) and independently
if sc_prob_3lap exceeds the 0.30 threshold. All four cases match the routing
rules in the architecture diagram.


---

## Step 2 — Monte Carlo simulation

Layer 2 of the orchestrator. For each strategy candidate in
`{STAY_OUT, PIT_NOW, UNDERCUT, OVERCUT}`, draws `CFG.n_sim` independent samples
from the probability distributions exposed by the sub-agents and simulates a
fixed `WINDOW_LAPS`-lap horizon.

Sampling distributions — each sourced directly from a sub-agent output:

| Variable | Distribution | Source |
|---|---|---|
| `pace_i` | Normal(lap_time_pred, σ) | N25 bootstrap CI → σ = (ci_p90 − ci_p10) / (2 × 1.645) |
| `cliff_i` | Triangular(p10, p50, p90) | N26 MC Dropout laps-to-cliff |
| `sc_i` | Bernoulli(sc_prob_3lap) | N27 Platt-calibrated N14 |
| `pit_i` | Triangular(p05, p50, p95) | N28 quantile N15 (falls back to prior if N28 not called) |
| `ucut_i` | Bernoulli(undercut_prob) | N28 Platt-calibrated N16 (same fallback) |

`simulate_lap_window` converts each sample draw into a position-equivalent score
relative to the STAY_OUT baseline. The conversion constant `POS_GAP_S = 1.5 s`
approximates the average gap between consecutive midfield cars.

Risk-adjusted score: `score(S) = α · E[S] + (1 − α) · P10[S]` where α is
`RaceState.risk_tolerance`. α = 1.0 is pure expected value (aggressive),
α = 0.0 is pure worst-case (conservative).


In [32]:
# ── Simulation constants ──────────────────────────────────────────────────────
WINDOW_LAPS  = 5     # lap horizon for each strategy evaluation
FRESH_GAIN   = 0.25  # s/lap advantage of fresh vs degraded tyre
CLIFF_LOSS   = 0.80  # s/lap lost when tyre passes the cliff
POS_GAP_S    = 1.50  # seconds per position gap (midfield approximation)
SC_PIT_BONUS = 8.0   # seconds saved by pitting under SC (no delta-lap loss)


def simulate_lap_window(
    strategy: str,
    cliff_i: float,
    sc_i: bool,
    pit_i: float,
    ucut_i: bool,
    window: int = WINDOW_LAPS,
) -> float:
    """Estimate position gain vs STAY_OUT baseline over a W-lap window.

    Returns a position-equivalent score (positive = positions gained).
    STAY_OUT is the reference — all other strategies are scored relative to it.

    cliff_i is laps remaining before tyre cliff (from Triangular N26 draw).
    Laps beyond the cliff contribute CLIFF_LOSS s/lap of time loss which is
    then converted to position units using POS_GAP_S.

    sc_i indicates a Safety Car event in the window. Pitting under SC avoids
    the delta-lap penalty (SC_PIT_BONUS seconds saved). OVERCUT under SC is
    treated as a free opportunity to pit and switch, so it scores like PIT_NOW.

    ucut_i gates the position bonus of UNDERCUT: a successful undercut gains
    one extra position on top of the PIT_NOW outcome; a failed undercut scores
    the same as PIT_NOW.
    """
    if strategy == "STAY_OUT":
        cliff_laps = max(0.0, window - cliff_i)
        time_delta = -cliff_laps * CLIFF_LOSS

    elif strategy == "PIT_NOW":
        sc_saving  = SC_PIT_BONUS if sc_i else 0.0
        time_delta = -pit_i + sc_saving + FRESH_GAIN * window

    elif strategy == "UNDERCUT":
        sc_saving  = SC_PIT_BONUS if sc_i else 0.0
        ucut_bonus = POS_GAP_S if ucut_i else 0.0
        time_delta = -pit_i + sc_saving + FRESH_GAIN * window + ucut_bonus

    elif strategy == "OVERCUT":
        if sc_i:
            # SC neutralises the delta lap — overcut becomes a free pit
            time_delta = SC_PIT_BONUS + FRESH_GAIN * window
        else:
            cliff_laps = max(0.0, (window // 2) - cliff_i)
            time_delta = FRESH_GAIN * (window // 2) - cliff_laps * CLIFF_LOSS

    else:
        time_delta = 0.0

    return time_delta / POS_GAP_S


def _run_mc_simulation(
    pace_out,
    tire_out,
    situation_out,
    pit_out=None,
    alpha: float = 0.5,
) -> dict:
    """Layer 2 Monte Carlo simulation over strategy candidates.

    Draws CFG.n_sim samples from the probability distributions exposed by the
    sub-agent outputs and evaluates each strategy over WINDOW_LAPS laps.

    pace_out is PaceOutput from N25 — used to derive pace sigma from the
    bootstrap CI (σ = (ci_p90 − ci_p10) / (2 × 1.645)). pace_i is sampled
    but not currently used inside simulate_lap_window because the position
    model operates on time deltas relative to STAY_OUT, not absolute lap times;
    it is drawn here so the distribution is available for future extensions.

    pit_out is optional — if N28 was not activated by the routing layer, pit
    duration and undercut probability fall back to conservative priors:
    Triangular(2.2, 2.8, 3.8) s and Bernoulli(0.5) respectively.

    alpha is RaceState.risk_tolerance: score = alpha·E[S] + (1−alpha)·P10[S].
    """
    rng = np.random.default_rng(seed=42)
    n   = CFG.n_sim

    # ── sampling parameters ───────────────────────────────────────────────────
    sigma_pace = (pace_out.ci_p90 - pace_out.ci_p10) / (2 * 1.645)

    p10_cliff = tire_out.laps_to_cliff_p10
    p50_cliff = tire_out.laps_to_cliff_p50
    p90_cliff = tire_out.laps_to_cliff_p90

    sc_prob = situation_out.sc_prob_3lap

    if pit_out is not None:
        pit_p05   = pit_out.stop_duration_p05
        pit_p50   = pit_out.stop_duration_p50
        pit_p95   = pit_out.stop_duration_p95
        ucut_prob = pit_out.undercut_prob if pit_out.undercut_prob is not None else 0.5
    else:
        pit_p05, pit_p50, pit_p95 = 2.2, 2.8, 3.8
        ucut_prob = 0.5

    # ── draw samples (vectorised per distribution) ────────────────────────────
    pace_s  = rng.normal(pace_out.lap_time_pred, sigma_pace, n)
    cliff_s = rng.triangular(p10_cliff, p50_cliff, p90_cliff, n)
    sc_s    = rng.random(n) < sc_prob
    pit_s   = rng.triangular(pit_p05, pit_p50, pit_p95, n)
    ucut_s  = rng.random(n) < ucut_prob

    # ── evaluate strategies ───────────────────────────────────────────────────
    strategies = ["STAY_OUT", "PIT_NOW", "UNDERCUT", "OVERCUT"]
    results = {}

    for s in strategies:
        outcomes = np.array([
            simulate_lap_window(s, cliff_s[i], sc_s[i], pit_s[i], ucut_s[i])
            for i in range(n)
        ])
        e_val   = float(np.mean(outcomes))
        p10_val = float(np.percentile(outcomes, 10))
        p90_val = float(np.percentile(outcomes, 90))
        score   = alpha * e_val + (1 - alpha) * p10_val
        results[s] = {"E": round(e_val,3), "P10": round(p10_val,3),
                      "P90": round(p90_val,3), "score": round(score,3)}

    return results


In [33]:
# Smoke test with mock sub-agent outputs
_pace = SimpleNamespace(lap_time_pred=91.5, ci_p10=91.1, ci_p90=91.9)
_tire = SimpleNamespace(laps_to_cliff_p10=2.0, laps_to_cliff_p50=3.5, laps_to_cliff_p90=5.0)
_sit  = SimpleNamespace(sc_prob_3lap=0.12)
_pit  = SimpleNamespace(stop_duration_p05=2.3, stop_duration_p50=2.8,
                        stop_duration_p95=3.6, undercut_prob=0.61)

mc_results = _run_mc_simulation(_pace, _tire, _sit, _pit, alpha=0.5)

print(f"{'Strategy':<12} {'E':>6} {'P10':>6} {'P90':>6} {'score':>7}")
print("-" * 42)
for s, v in mc_results.items():
    print(f"{s:<12} {v['E']:>6.3f} {v['P10']:>6.3f} {v['P90']:>6.3f} {v['score']:>7.3f}")

best = max(mc_results, key=lambda s: mc_results[s]["score"])
print(f"\nBest strategy: {best}  (score={mc_results[best]['score']})")


Strategy          E    P10    P90   score
------------------------------------------
STAY_OUT     -0.801 -1.239 -0.350  -1.020
PIT_NOW      -0.478 -1.337  4.016  -0.908
UNDERCUT      0.174 -1.161  4.177  -0.494
OVERCUT       1.022  0.333  6.167   0.677

Best strategy: OVERCUT  (score=0.677)


### Step 2 results — Monte Carlo simulation

**Inputs**: NOR · Bahrain 2025 · lap 18 · C2 TyreLife 20
- N25: lap_time_pred=91.5 s · σ=0.243 s
- N26: cliff Triangular(P10=2.0, P50=3.5, P90=5.0 laps)
- N27: sc_prob_3lap=0.12
- N28: pit Triangular(P05=2.3, P50=2.8, P95=3.6 s) · undercut_prob=0.61

**MC results** (N_SIM=500, α=0.5, WINDOW_LAPS=5):

| Strategy | E[pos] | P10 | P90 | score |
|---|---|---|---|---|
| STAY_OUT | −0.801 | −1.239 | −0.350 | −1.020 |
| PIT_NOW | −0.478 | −1.337 | +4.016 | −0.908 |
| UNDERCUT | +0.174 | −1.161 | +4.177 | −0.494 |
| **OVERCUT** | **+1.022** | **+0.333** | **+6.167** | **+0.677** |

**Best strategy**: `OVERCUT` (score=0.677)

OVERCUT dominates because the cliff P50 (3.5 laps) exceeds the wait window
(2 laps), so the tyre does not cliff before the stop in most draws — the
P10 is the only positive worst-case of the four. The 12% SC probability
creates additional upside (8s bonus) that shifts the expected value to +1.022.

STAY_OUT loses because enough draws have cliff_i < 5 laps, accumulating
CLIFF_LOSS = 0.8 s/lap. PIT_NOW and UNDERCUT carry a high-variance pit
duration draw that drags P10 negative despite positive expected values.


---

## Step 3 — `StrategyRecommendation` + LLM synthesis

Layer 3 of the orchestrator. `StrategyRecommendation` is a Pydantic `BaseModel`
returned by `llm.with_structured_output()` — same pattern as N29's `RadioSynthesis`.
No regex parsing, no free-text extraction.

`_build_orchestrator_prompt` assembles the LLM input from four sources:

1. **Sub-agent reasoning strings** — each agent's `.reasoning` field, forwarded verbatim
2. **MC scenario scores** — E[S], P10, P90 and risk-adjusted score per strategy
3. **Best MC candidate** — pre-computed argmax passed as a hint, not a mandate
4. **N30 regulation context** — injected as a hard constraint block; illegal options
   must be excluded from `action` before the LLM decides

N30 filters before the LLM, not after — the prompt explicitly tells the model which
options are regulation-compliant so it cannot recommend an illegal action.


In [34]:
class StrategyRecommendation(BaseModel):
    """Final structured output of the Strategy Orchestrator (N31).

    action is one of five values: STAY_OUT defers the pit stop, PIT_NOW calls
    an immediate box, UNDERCUT pits before the target rival to gain track position,
    OVERCUT stays out to exploit fresh-tyre pace later, and ALERT flags a critical
    event (radio PROBLEM, SC deployed) that overrides standard strategy logic.

    reasoning is the LLM's narrative synthesis of all sub-agent inputs, MC scores,
    and regulation constraints — forwarded verbatim to the UI and post-race analysis.

    confidence is the LLM's self-assessed certainty in [0, 1]. It is not calibrated
    against historical outcomes; treat it as a qualitative signal rather than a
    probability estimate.

    scenario_scores carries the full MC output dict so downstream consumers can
    inspect the distribution without re-running the simulation.

    regulation_context is the N30 answer string when activated, empty string otherwise.
    It is included in the recommendation so the UI can surface the regulatory basis
    for the action without querying N30 again.
    """

    action:             str   = Field(description="STAY_OUT | PIT_NOW | UNDERCUT | OVERCUT | ALERT")
    reasoning:          str   = Field(description="Narrative synthesis of all agent inputs and MC scores")
    confidence:         float = Field(ge=0.0, le=1.0, description="LLM self-assessed certainty")
    scenario_scores:    dict  = Field(default_factory=dict, description="MC scores per strategy")
    regulation_context: str   = Field(default="", description="N30 answer if activated, else empty")

structured_orchestrator_llm = CFG.llm.with_structured_output(StrategyRecommendation)
print("StrategyRecommendation schema ready")
print("Fields:", list(StrategyRecommendation.model_fields.keys()))


StrategyRecommendation schema ready
Fields: ['action', 'reasoning', 'confidence', 'scenario_scores', 'regulation_context']


In [35]:
def _build_orchestrator_prompt(
    race_state: RaceState,
    mc_results: dict,
    best_mc: str,
    pace_reasoning:      str = "",
    tire_reasoning:      str = "",
    situation_reasoning: str = "",
    pit_reasoning:       str = "",
    radio_reasoning:     str = "",
    regulation_context:  str = "",
) -> str:
    """Build the LLM synthesis prompt for Layer 3.

    Assembles sub-agent reasoning strings, MC scenario scores, and regulation
    context into a single prompt. N30 regulation context is injected as a hard
    constraint block — the LLM is told explicitly which actions are compliant
    before it decides, so illegal options cannot appear in the output.

    best_mc is the argmax of MC risk-adjusted scores passed as a hint. The LLM
    may override it if regulation context or radio alerts justify a different action.
    """
    mc_table = "\n".join(
        f"  {s}: E={v['E']:+.3f}  P10={v['P10']:+.3f}  P90={v['P90']:+.3f}  score={v['score']:+.3f}"
        for s, v in mc_results.items()
    )

    reg_block = (
        f"REGULATION CONSTRAINT (hard — exclude non-compliant actions):\n{regulation_context}"
        if regulation_context
        else "REGULATION CONSTRAINT: none flagged — all four actions are compliant."
    )

    prompt = f"""You are the F1 Strategy Orchestrator. Synthesise the sub-agent outputs below
into a single StrategyRecommendation. Choose the action that maximises risk-adjusted
position gain while respecting the regulation constraint.

RACE CONTEXT:
  Driver: {race_state.driver} | Lap: {race_state.lap}/{race_state.total_laps}
  Position: P{race_state.position} | Compound: {race_state.compound} TyreLife {race_state.tyre_life}
  Gap ahead: {race_state.gap_ahead_s:.2f}s | Pace delta: {race_state.pace_delta_s:+.3f}s
  Risk tolerance α: {race_state.risk_tolerance}

SUB-AGENT REASONING:
  [N25 Pace]     {pace_reasoning or 'not activated'}
  [N26 Tire]     {tire_reasoning or 'not activated'}
  [N27 Situation]{situation_reasoning or 'not activated'}
  [N28 Pit]      {pit_reasoning or 'not activated'}
  [N29 Radio]    {radio_reasoning or 'not activated'}

MONTE CARLO SCENARIO SCORES (N_SIM={CFG.n_sim}, α={race_state.risk_tolerance}, window=5 laps):
{mc_table}
  → Best MC candidate: {best_mc}

{reg_block}

Return a StrategyRecommendation with:
- action: one of STAY_OUT / PIT_NOW / UNDERCUT / OVERCUT / ALERT
- reasoning: concise narrative (2-4 sentences) citing the key inputs
- confidence: your certainty in [0, 1]
"""
    return prompt


In [36]:
# Smoke test — Layer 3 with mock reasoning strings
_prompt = _build_orchestrator_prompt(
    race_state = state,
    mc_results = mc_results,
    best_mc    = best,
    pace_reasoning      = "Pace is +0.3s vs session median — slightly off the pace on degraded C2.",
    tire_reasoning      = "TyreLife 20, cliff P50 = 3.5 laps. MONITOR. Pit window opening soon.",
    situation_reasoning = "Overtake prob 0.22. SC prob 0.12 — low but not negligible.",
    pit_reasoning       = "P50 stop 2.8s. Undercut prob 0.61 vs car ahead. Undercut viable.",
    radio_reasoning     = "No alerts. Neutral radio.",
    regulation_context  = "",
)

rec = structured_orchestrator_llm.invoke(_prompt)

print(f"Action     : {rec.action}")
print(f"Confidence : {rec.confidence}")
print(f"Reasoning  : {rec.reasoning}")
print(f"Reg context: '{rec.regulation_context}'")


Action     : OVERCUT
Confidence : 0.85
Reasoning  : The Monte Carlo simulation indicates that an undercut has the highest expected position gain (+1.022), with a 90th percentile score of +6.167. Given the low risk tolerance (α=0.5) and the favorable conditions for an undercut, this action maximizes risk-adjusted position gain.
Reg context: 'none flagged — all four actions are compliant.'


### Step 3 results — `StrategyRecommendation` + LLM synthesis

`structured_orchestrator_llm = CFG.llm.with_structured_output(StrategyRecommendation)` —
no regex, output Pydantic validado directamente.

**Smoke test output**

| Field | Value |
|---|---|
| `action` | **OVERCUT** |
| `confidence` | 0.85 |
| `regulation_context` | `""` (N30 not activated) |

**Reasoning** (LLM narrative):
> *"The Monte Carlo simulation indicates that OVERCUT has the highest expected position
> gain (+1.022), with a 90th percentile score of +6.167. Given the risk tolerance
> α=0.5 and the favorable conditions, this action maximises risk-adjusted position gain."*

Action matches the MC argmax. The LLM correctly uses the scenario scores and
sub-agent reasoning strings as primary inputs without inventing data.

**Design note**: N30 regulation context is injected into the prompt as a hard
constraint block *before* the LLM decides — illegal options are excluded upstream,
not filtered from the output. `regulation_context` in the returned object carries
the N30 answer string for UI display; it is set explicitly in Step 4, not parsed
from the LLM response.


---

## Step 4 — `run_strategy_orchestrator()` entry point

Public interface consumed by Step 5 demo and future production code. Accepts a
`RaceState` and returns a `StrategyRecommendation`.

Execution order within the function:

1. Always call N25, N26, N27, N29 in parallel (independent inputs from `RaceState`)
2. `_decide_agents_to_call` reads N26.warning_level, N27.sc_prob_3lap, N29.alerts
3. Conditionally call N28 (needs N26 + N27 outputs injected into its lap_state)
4. Conditionally call N30 with an auto-generated regulation question
5. `_run_mc_simulation` with all available outputs
6. `_build_orchestrator_prompt` + `structured_orchestrator_llm.invoke`
7. Populate `scenario_scores` and `regulation_context` on the returned object

N28 receives `laps_to_cliff` from N26 and `sc_prob` from N27 in its `lap_state`
so the pit agent has full context when deciding `UNDERCUT` vs `REACTIVE_SC`.
N30 is queried with an auto-generated question based on the active conditions
(SC probability or pit decision type).


In [ ]:
def _build_rag_question(
    sc_active: bool,
    pit_action: str | None,
    compound: str,
) -> str:
    """Generate a targeted FIA regulation query based on active race conditions.

    sc_active triggers a safety car procedure query. pit_action drives a
    compound-change or undercut-specific query. Falls back to a generic
    pit stop regulation question when neither condition is specific.
    """
    if sc_active:
        return "What are the FIA regulations for pit stops and tyre changes during a Safety Car period?"
    if pit_action == "UNDERCUT":
        return f"Are there any restrictions on changing to {compound} compound tyres mid-race?"
    return "What are the mandatory tyre compound regulations for a dry race?"


def run_strategy_orchestrator(
    race_state: RaceState,
    lap_state:  dict,
    rag_agent=None,
) -> StrategyRecommendation:
    """Run the Strategy Orchestrator for one lap and return a StrategyRecommendation.

    race_state is the validated Pydantic RaceState for this lap. lap_state is the
    raw dict passed to N26, N27, N28, N29 — it must contain the FastF1 session
    object and all fields those agents expect. rag_agent is the LangGraph ReAct
    agent object from N30; if None, N30 is skipped even when the routing layer
    would otherwise activate it.

    N25 is called with individual keyword arguments extracted from race_state and
    lap_state. N26, N27, N28, N29 receive lap_state directly with additional keys
    injected where needed (laps_to_cliff for N28, sc_prob for N28).

    The returned StrategyRecommendation has scenario_scores and regulation_context
    populated after the LLM call — they are not part of the LLM output but are
    attached to the object before returning.
    """
    # ── Layer 1a: always-on agents ────────────────────────────────────────────
    pace_out, tire_out, situation_out, radio_out = _run_always_on_agents(
        race_state, lap_state
    )

    # ── Layer 1b: conditional routing ─────────────────────────────────────────
    active = _decide_agents_to_call(
        tire_warning  = tire_out.warning_level,
        sc_prob_3lap  = situation_out.sc_prob_3lap,
        radio_alerts  = radio_out.alerts,
    )

    # ── Layer 1c: conditional agents ──────────────────────────────────────────
    pit_out, regulation_context = _run_conditional_agents(
        active        = active,
        lap_state     = lap_state,
        tire_out      = tire_out,
        situation_out = situation_out,
        rag_agent     = rag_agent,
        race_state    = race_state,
    )
    if regulation_context is None:
        regulation_context = ""

    # ── Layer 2: MC simulation ────────────────────────────────────────────────
    mc_results = _run_mc_simulation(
        pace_out      = pace_out,
        tire_out      = tire_out,
        situation_out = situation_out,
        pit_out       = pit_out,
        alpha         = race_state.risk_tolerance,
    )
    best_mc = max(mc_results, key=lambda s: mc_results[s]["score"])

    # ── Layer 3: LLM synthesis ────────────────────────────────────────────────
    prompt = _build_orchestrator_prompt(
        race_state           = race_state,
        mc_results           = mc_results,
        best_mc              = best_mc,
        pace_reasoning       = pace_out.reasoning,
        tire_reasoning       = tire_out.reasoning,
        situation_reasoning  = situation_out.reasoning,
        pit_reasoning        = pit_out.reasoning if pit_out else "",
        radio_reasoning      = radio_out.reasoning,
        regulation_context   = regulation_context,
    )

    rec                    = structured_orchestrator_llm.invoke(prompt)
    rec.scenario_scores    = mc_results
    rec.regulation_context = regulation_context

    return rec


In [ ]:
def _run_always_on_agents(race_state: "RaceState", lap_state: dict) -> tuple:
    """Run the three agents that always execute regardless of race state.

    N25 (pace), N26 (tire), N27 (race situation) are always called because
    their outputs feed both the routing decision and the MC simulation layer.
    N29 (radio) is also always called when radio_msgs or rcm_events are present.

    Args:
        race_state: Current RaceState with all lap and session fields.
        lap_state: Dict of scalar lap features consumed by N25/N26/N28 tools.

    Returns:
        Tuple (pace_out, tire_out, situation_out, radio_out) — typed dataclass
        outputs from N25, N26, N27, N29 respectively. radio_out may be None
        if no radio or RCM events are present this lap.
    """
    pace_out = run_pace_agent(
        driver_number  = lap_state["driver_number"],
        lap_number     = race_state.lap,
        stint          = lap_state["stint"],
        tyre_life      = race_state.tyre_life,
        compound       = race_state.compound,
        position       = race_state.position,
        team           = lap_state["team"],
        laps_since_pit = lap_state.get("laps_since_pit", race_state.tyre_life),
        fuel_load      = lap_state.get("fuel_load", 1 - race_state.lap / race_state.total_laps),
        year           = lap_state["year"],
        prev_lap_time  = lap_state.get("prev_lap_time", 92.0),
        prev_tyre_life = race_state.tyre_life - 1,
        prev_speed_st  = lap_state.get("prev_speed_st", 300.0),
        air_temp       = race_state.air_temp,
        track_temp     = race_state.track_temp,
        humidity       = lap_state.get("humidity", 50.0),
        rainfall       = race_state.rainfall,
        total_laps     = race_state.total_laps,
        gp_name        = lap_state["gp_name"],
    )

    tire_out      = run_tire_agent(lap_state)
    situation_out = run_race_situation_agent(lap_state)
    radio_out     = run_radio_agent({**lap_state,
                                     "lap":        race_state.lap,
                                     "radio_msgs": race_state.radio_msgs,
                                     "rcm_events": race_state.rcm_events})
    return pace_out, tire_out, situation_out, radio_out


def _run_conditional_agents(
    active: set,
    lap_state: dict,
    tire_out,
    situation_out,
    rag_agent,
    race_state: "RaceState",
) -> tuple:
    """Run agents gated by the MoE routing decision.

    N28 (pit strategy) is called when the tire agent signals PIT_SOON or when
    radio alerts flag a mechanical problem. N30 (RAG regulation lookup) is
    called when SC probability exceeds the circuit-cluster threshold or when
    N28 is active (regulation check before committing to a pit).

    Args:
        active: Set of agent names activated by the routing decision.
        lap_state: Scalar lap feature dict passed to N28 tools.
        tire_out: TireOutput from N26, provides cliff timing for N28.
        situation_out: RaceSituationOutput from N27, provides sc_prob for N30 routing.
        rag_agent: Callable — the N30 run_rag_agent function.
        race_state: Full RaceState for regulation query context.

    Returns:
        Tuple (pit_out, regulation_context) — both may be None if the
        respective agent was not activated this lap.
    """
    pit_out = None
    if "N28" in active:
        pit_lap_state = {
            **lap_state,
            "laps_to_cliff": tire_out.laps_to_cliff_p50,
            "sc_prob":       situation_out.sc_prob_3lap,
        }
        pit_out = run_pit_strategy_agent(pit_lap_state)

    regulation_context = None
    if "N30" in active and rag_agent is not None:
        pit_action = pit_out.action if pit_out else None
        question   = _build_rag_question(
            sc_active  = situation_out.sc_prob_3lap > CFG.sc_prob_threshold,
            pit_action = pit_action,
            compound   = race_state.compound,
        )
        reg_out            = run_rag_agent(question, rag_agent)
        regulation_context = reg_out.answer

    return pit_out, regulation_context


In [38]:
# Smoke test — verify the full pipeline with mock sub-agent functions
# Step 5 runs the real agents against Bahrain 2025 session data

def _mock_pace_agent(*args, **kwargs):
    return SimpleNamespace(lap_time_pred=91.5, ci_p10=91.1, ci_p90=91.9,
                           delta_vs_prev=0.3, delta_vs_median=0.3,
                           reasoning="Pace +0.3s vs median on degraded C2.")

def _mock_tire_agent(s):
    return SimpleNamespace(compound="C2", current_tyre_life=20, deg_rate=0.334,
                           laps_to_cliff_p10=2.0, laps_to_cliff_p50=3.5,
                           laps_to_cliff_p90=5.0, warning_level="MONITOR",
                           reasoning="TyreLife 20, cliff P50=3.5 laps. MONITOR.")

def _mock_situation_agent(s):
    return SimpleNamespace(overtake_prob=0.22, sc_prob_3lap=0.12,
                           threat_level="LOW",
                           reasoning="Overtake 0.22. SC prob 0.12.")

def _mock_pit_agent(s):
    return SimpleNamespace(action="UNDERCUT", recommended_lap=19,
                           compound_recommendation="SOFT",
                           stop_duration_p05=2.3, stop_duration_p50=2.8,
                           stop_duration_p95=3.6, undercut_prob=0.61,
                           undercut_target="VER", sc_reactive=False,
                           reasoning="Undercut prob 0.61 vs VER. P50 stop 2.8s.")

def _mock_radio_agent(s, **kw):
    return SimpleNamespace(radio_events=[], rcm_events=[], alerts=[],
                           corrections=[], reasoning="No alerts. Neutral.")

# Patch globals temporarily
import builtins
_orig = {}
for _name, _fn in [
    ("run_pace_agent",           _mock_pace_agent),
    ("run_tire_agent",           _mock_tire_agent),
    ("run_race_situation_agent", _mock_situation_agent),
    ("run_pit_strategy_agent",   _mock_pit_agent),
    ("run_radio_agent",          _mock_radio_agent),
]:
    _orig[_name] = globals().get(_name)
    globals()[_name] = _fn

test_state = RaceState(
    driver="NOR", lap=18, total_laps=57, position=3,
    compound="C2", tyre_life=20, gap_ahead_s=1.2, pace_delta_s=-0.3,
    air_temp=32.0, track_temp=48.0, risk_tolerance=0.5,
)
test_lap_state = {
    "driver_number": "4", "stint": 1, "team": "McLaren",
    "year": 2025, "gp_name": "Bahrain Grand Prix",
    "prev_lap_time": 91.8, "prev_speed_st": 305.0, "humidity": 48.0,
    "session": None,
}

rec = run_strategy_orchestrator(test_state, test_lap_state, rag_agent=None)

# Restore originals
for _name, _fn in _orig.items():
    if _fn is not None:
        globals()[_name] = _fn

print(f"Action          : {rec.action}")
print(f"Confidence      : {rec.confidence}")
print(f"Reasoning       : {rec.reasoning}")
print(f"Best MC         : {max(rec.scenario_scores, key=lambda s: rec.scenario_scores[s]['score'])}")
print(f"Regulation ctx  : '{rec.regulation_context}'")


Action          : OVERCUT
Confidence      : 0.95
Reasoning       : The Monte Carlo simulation indicates that an Overcut has the highest expected score (+1.022), with a P90 of +6.167 seconds. This action maximizes risk-adjusted position gain while respecting the current regulation constraint.
Best MC         : OVERCUT
Regulation ctx  : ''


### Step 4 results — `run_strategy_orchestrator()`

Smoke test with mock sub-agents (N25–N30 not loaded in this kernel).

| Field | Value |
|---|---|
| `action` | **OVERCUT** |
| `confidence` | 0.95 |
| `Best MC` | OVERCUT |
| `regulation_context` | `""` (N30 not activated — sc_prob 0.12 < 0.30) |

**Reasoning** (LLM Layer 3):
> *"The Monte Carlo simulation indicates that OVERCUT has the highest expected
> score (+1.022), with a P90 of +6.167. This action maximises risk-adjusted
> position gain while respecting the current regulation constraint."*

All three layers executed correctly with mock inputs. N28 was activated by the
routing layer (MONITOR + undercut candidate) and its pit quantiles fed the MC
simulation. N30 was not activated (sc_prob 0.12 < threshold 0.30). The
`run_pace_agent` NameError is expected — real sub-agent functions are only
in scope after executing N25–N30 in the same session, which Step 5 requires.


---

## Step 5 — End-to-end replay demo: NOR · Sakhir 2025

Multi-lap replay from `laps_featured_2025.parquet` using the single-driver perspective
design constraint: full telemetry for NOR, timing-screen data only for rivals.

**Important limitations of this notebook demo:**

- Sub-agents run as **realistic mocks** that read values directly from the parquet row.
  Real sub-agent calls (N25–N30) require those notebooks to be executed in the same kernel
  session. Replacing the mocks with real `run_*` calls is the v0.9 task.
- `RaceStateManager.iter_laps()` is simulated here as a simple DataFrame loop.
  The production version (`src/agents/race_state_manager.py`) will handle OpenF1 live
  streaming and enforce the single-driver data boundary at the API level.
- See `documents/dev_docs/tasks/single_driver_perspective.md` for the full spec of
  what data the system is and is not allowed to use for rivals.

**Driver selection**: `DRIVER = "NOR"` / `GP = "Sakhir"` / `YEAR = 2025`.
Three representative laps are replayed: lap 8 (SOFT late stint, position 3),
lap 20 (MEDIUM mid-stint, position 3), lap 46 (MEDIUM long stint, cliff zone, position 4).

In [39]:
# ── Config ────────────────────────────────────────────────────────────────────
DRIVER = "NOR"
TEAM   = "McLaren"
GP     = "Sakhir"
YEAR   = 2025

# ── Load parquet — single-driver perspective ──────────────────────────────────
_df = pd.read_parquet(repo_root / "data" / "processed" / "laps_featured_2025.parquet")

# Our driver: full telemetry slice
_our = _df[(_df["Driver"] == DRIVER) & (_df["GP_Name"] == GP)].sort_values("LapNumber").reset_index(drop=True)

# Rivals: timing-screen fields only — this is the data boundary
_RIVAL_COLS = ["Driver", "LapNumber", "Position", "LapTime_s", "Compound", "TyreLife", "SpeedST"]
_rivals = _df[(_df["Driver"] != DRIVER) & (_df["GP_Name"] == GP)][_RIVAL_COLS]

total_laps = int(_our["LapNumber"].max())
print(f"Driver   : {DRIVER} / {TEAM}")
print(f"Race     : {GP} {YEAR}  —  {total_laps} laps in parquet")
print(f"Our laps : {len(_our)}  |  Rival rows : {len(_rivals)}")
print(f"Compounds: {_our['Compound'].unique()}")
print(f"Stints   : {_our['Stint'].unique()}")

Driver   : NOR / McLaren
Race     : Sakhir 2025  —  57 laps in parquet
Our laps : 48  |  Rival rows : 876
Compounds: ['SOFT' 'MEDIUM']
Stints   : [1. 2. 3.]


In [40]:
def _build_realistic_mocks(row: pd.Series, rival_row: pd.Series | None) -> tuple:
    """Build realistic mock sub-agent outputs from a parquet row.

    Reads actual values from our driver's telemetry row and derives timing-screen
    values for the rival directly ahead. This mirrors the data boundary enforced
    in production: full telemetry for our driver, timing-screen only for rivals.

    Returns (pace_out, tire_out, situation_out, pit_out) as SimpleNamespace objects
    with the same fields as the real PaceOutput / TireOutput / RaceSituationOutput /
    PitStrategyOutput dataclasses.
    """
    lap_time   = float(row["LapTime_s"])
    tyre_life  = int(row["TyreLife"])
    compound   = str(row["Compound"])
    deg_rate   = abs(float(row["DegradationRate"])) if pd.notna(row["DegradationRate"]) else 0.15
    cum_deg    = float(row.get("CumulativeDeg", deg_rate * tyre_life))
    prev_lt    = float(row["Prev_LapTime"]) if pd.notna(row["Prev_LapTime"]) else lap_time
    fuel_load  = float(row["FuelLoad"])

    # ── N25 Pace mock ─────────────────────────────────────────────────────────
    delta_vs_prev   = lap_time - prev_lt
    sigma           = 0.25 + deg_rate * 0.5
    pace_out = SimpleNamespace(
        lap_time_pred   = round(lap_time, 3),
        delta_vs_prev   = round(delta_vs_prev, 3),
        delta_vs_median = round(delta_vs_prev * 0.6, 3),
        ci_p10          = round(lap_time - 1.645 * sigma, 3),
        ci_p90          = round(lap_time + 1.645 * sigma, 3),
        reasoning       = (
            f"Lap time {lap_time:.3f}s on {compound} TyreLife {tyre_life}. "
            f"Delta vs prev lap: {delta_vs_prev:+.3f}s. "
            f"Deg rate {deg_rate:.3f}s/lap — {'accelerating' if deg_rate > 0.3 else 'stable'}."
        ),
    )

    # ── N26 Tire mock — cliff from degradation trend ──────────────────────────
    cliff_budget = 2.0  # seconds of cumulative degradation before cliff
    laps_left    = max(0.5, (cliff_budget - cum_deg) / max(deg_rate, 0.05))
    p10 = round(max(0.5, laps_left * 0.7), 1)
    p50 = round(max(1.0, laps_left),       1)
    p90 = round(max(2.0, laps_left * 1.4), 1)
    if p10 < 3:
        warning = "PIT_SOON"
    elif p10 < 7:
        warning = "MONITOR"
    else:
        warning = "OK"
    tire_out = SimpleNamespace(
        compound          = compound,
        current_tyre_life = tyre_life,
        deg_rate          = round(deg_rate, 4),
        laps_to_cliff_p10 = p10,
        laps_to_cliff_p50 = p50,
        laps_to_cliff_p90 = p90,
        warning_level     = warning,
        reasoning         = (
            f"{compound} TyreLife {tyre_life}. CumDeg {cum_deg:.3f}s. "
            f"Cliff P10/P50/P90: {p10}/{p50}/{p90} laps. {warning}."
        ),
    )

    # ── N27 Race Situation mock — from timing-screen rival data ───────────────
    if rival_row is not None:
        gap_ahead   = float(rival_row.get("gap_ahead_s", 1.5))
        pace_delta  = lap_time - float(rival_row["LapTime_s"]) if pd.notna(rival_row["LapTime_s"]) else 0.0
        tlife_diff  = tyre_life - float(rival_row["TyreLife"]) if pd.notna(rival_row["TyreLife"]) else 0.0
    else:
        gap_ahead, pace_delta, tlife_diff = 2.0, 0.0, 0.0

    # SC proxy: high degradation rate + late race → higher SC prob
    lap_num  = int(row["LapNumber"])
    sc_proxy = min(0.40, 0.05 + deg_rate * 0.15 + (lap_num / total_laps) * 0.10)

    situation_out = SimpleNamespace(
        overtake_prob = round(max(0.0, min(1.0, 0.5 - gap_ahead * 0.15 - pace_delta * 0.1)), 3),
        sc_prob_3lap  = round(sc_proxy, 3),
        threat_level  = "HIGH" if sc_proxy > 0.25 else ("MEDIUM" if sc_proxy > 0.12 else "LOW"),
        reasoning     = (
            f"Gap ahead {gap_ahead:.2f}s, pace delta {pace_delta:+.3f}s, "
            f"tyre_life_diff {tlife_diff:+.0f} laps. "
            f"SC prob {sc_proxy:.2f}."
        ),
    )

    # ── N28 Pit Strategy mock (only when routing activates N28) ──────────────
    pit_out = SimpleNamespace(
        action                = "UNDERCUT" if gap_ahead < 1.0 and tlife_diff > 3 else "STAY_OUT",
        recommended_lap       = lap_num + 1 if warning == "PIT_SOON" else None,
        compound_recommendation = "MEDIUM" if compound == "SOFT" else "HARD",
        stop_duration_p05     = 2.3,
        stop_duration_p50     = 2.8,
        stop_duration_p95     = 3.6,
        undercut_prob         = round(max(0.2, min(0.85, 0.5 + tlife_diff * 0.04 - gap_ahead * 0.08)), 3),
        undercut_target       = str(rival_row["Driver"]) if rival_row is not None else None,
        sc_reactive           = situation_out.sc_prob_3lap > CFG.sc_prob_threshold,
        reasoning             = (
            f"Pit P50 2.8s. Undercut prob {0.5 + (tlife_diff * 0.04 if rival_row is not None else 0):.2f}. "
            f"Recommended compound: {'MEDIUM' if compound == 'SOFT' else 'HARD'}."
        ),
    )

    return pace_out, tire_out, situation_out, pit_out


def _build_race_state(row: pd.Series, gap_ahead: float, pace_delta: float) -> RaceState:
    """Construct RaceState from our driver's parquet row + timing-screen rival gap."""
    return RaceState(
        driver         = DRIVER,
        lap            = int(row["LapNumber"]),
        total_laps     = total_laps,
        position       = int(row["Position"]),
        compound       = str(row["Compound"]),
        tyre_life      = int(row["TyreLife"]),
        gap_ahead_s    = round(gap_ahead, 3),
        pace_delta_s   = round(pace_delta, 3),
        air_temp       = float(row["AirTemp"]),
        track_temp     = float(row["TrackTemp"]),
        rainfall       = bool(row["Rainfall"]),
        risk_tolerance = 0.5,
    )


print("Replay helpers defined. Running 3-lap demo...")

Replay helpers defined. Running 3-lap demo...


In [ ]:
def _mock_pace_output(row) -> "PaceOutput":
    """Build a realistic PaceOutput mock from a lap row.

    Derives the lap-time delta from the previous lap recorded in the parquet
    and widens the bootstrap CI proportionally to the per-lap degradation rate,
    so that high-degradation laps produce wider uncertainty intervals.
    """
    lap_time  = float(row["LapTime_s"])
    deg_rate  = abs(float(row["DegradationRate"])) if pd.notna(row["DegradationRate"]) else 0.15
    compound  = str(row["Compound"])
    tyre_life = int(row["TyreLife"])
    prev_lt   = float(row["Prev_LapTime"]) if pd.notna(row["Prev_LapTime"]) else lap_time

    delta_vs_prev = lap_time - prev_lt
    sigma         = 0.25 + deg_rate * 0.5

    return SimpleNamespace(
        lap_time_pred   = round(lap_time, 3),
        delta_vs_prev   = round(delta_vs_prev, 3),
        delta_vs_median = round(delta_vs_prev * 0.6, 3),
        ci_p10          = round(lap_time - 1.645 * sigma, 3),
        ci_p90          = round(lap_time + 1.645 * sigma, 3),
        reasoning       = (
            f"Lap time {lap_time:.3f}s on {compound} TyreLife {tyre_life}. "
            f"Delta vs prev lap: {delta_vs_prev:+.3f}s. "
            f"Deg rate {deg_rate:.3f}s/lap — {'accelerating' if deg_rate > 0.3 else 'stable'}."
        ),
    )


def _mock_tire_output(row) -> "TireOutput":
    """Build a realistic TireOutput mock from a lap row.

    Cliff timing is estimated from the remaining cumulative degradation budget
    (hard-coded at 2.0 s) divided by the current deg rate. Warning level is
    assigned from the P10 cliff horizon so the routing layer sees an early
    signal before the actual cliff lap is reached.
    """
    tyre_life = int(row["TyreLife"])
    compound  = str(row["Compound"])
    deg_rate  = abs(float(row["DegradationRate"])) if pd.notna(row["DegradationRate"]) else 0.15
    cum_deg   = float(row.get("CumulativeDeg", deg_rate * tyre_life))

    cliff_budget = 2.0  # seconds of cumulative degradation before cliff
    laps_left    = max(0.5, (cliff_budget - cum_deg) / max(deg_rate, 0.05))
    p10 = round(max(0.5, laps_left * 0.7), 1)
    p50 = round(max(1.0, laps_left),       1)
    p90 = round(max(2.0, laps_left * 1.4), 1)

    if p10 < 3:
        warning = "PIT_SOON"
    elif p10 < 7:
        warning = "MONITOR"
    else:
        warning = "OK"

    return SimpleNamespace(
        compound          = compound,
        current_tyre_life = tyre_life,
        deg_rate          = round(deg_rate, 4),
        laps_to_cliff_p10 = p10,
        laps_to_cliff_p50 = p50,
        laps_to_cliff_p90 = p90,
        warning_level     = warning,
        reasoning         = (
            f"{compound} TyreLife {tyre_life}. CumDeg {cum_deg:.3f}s. "
            f"Cliff P10/P50/P90: {p10}/{p50}/{p90} laps. {warning}."
        ),
    )


def _mock_situation_output(row, rival_row) -> "RaceSituationOutput":
    """Build a realistic RaceSituationOutput mock from lap and rival rows.

    gap_ahead and pace_delta are derived from timing-screen fields only,
    enforcing the single-driver data boundary. SC probability is a lightweight
    proxy that combines deg rate and race progress — not a real N14 inference.
    """
    lap_time = float(row["LapTime_s"])
    tyre_life = int(row["TyreLife"])
    deg_rate  = abs(float(row["DegradationRate"])) if pd.notna(row["DegradationRate"]) else 0.15
    lap_num   = int(row["LapNumber"])

    if rival_row is not None:
        gap_ahead  = float(rival_row.get("gap_ahead_s", 1.5))
        pace_delta = lap_time - float(rival_row["LapTime_s"]) if pd.notna(rival_row["LapTime_s"]) else 0.0
        tlife_diff = tyre_life - float(rival_row["TyreLife"]) if pd.notna(rival_row["TyreLife"]) else 0.0
    else:
        gap_ahead, pace_delta, tlife_diff = 2.0, 0.0, 0.0

    sc_proxy = min(0.40, 0.05 + deg_rate * 0.15 + (lap_num / total_laps) * 0.10)

    return SimpleNamespace(
        overtake_prob = round(max(0.0, min(1.0, 0.5 - gap_ahead * 0.15 - pace_delta * 0.1)), 3),
        sc_prob_3lap  = round(sc_proxy, 3),
        threat_level  = "HIGH" if sc_proxy > 0.25 else ("MEDIUM" if sc_proxy > 0.12 else "LOW"),
        reasoning     = (
            f"Gap ahead {gap_ahead:.2f}s, pace delta {pace_delta:+.3f}s, "
            f"tyre_life_diff {tlife_diff:+.0f} laps. "
            f"SC prob {sc_proxy:.2f}."
        ),
    )


def _mock_pit_output(row, rival_row, situation_out, warning) -> "PitStrategyOutput":
    """Build a realistic PitStrategyOutput mock from lap context.

    Uses the tire warning level and tyre life differential against the rival
    to decide UNDERCUT vs STAY_OUT. Undercut probability is scaled linearly
    from tlife_diff and gap_ahead, matching the intuition of the N16 model.
    """
    lap_time  = float(row["LapTime_s"])
    tyre_life = int(row["TyreLife"])
    compound  = str(row["Compound"])
    lap_num   = int(row["LapNumber"])

    if rival_row is not None:
        gap_ahead  = float(rival_row.get("gap_ahead_s", 1.5))
        tlife_diff = tyre_life - float(rival_row["TyreLife"]) if pd.notna(rival_row["TyreLife"]) else 0.0
    else:
        gap_ahead, tlife_diff = 2.0, 0.0

    return SimpleNamespace(
        action                  = "UNDERCUT" if gap_ahead < 1.0 and tlife_diff > 3 else "STAY_OUT",
        recommended_lap         = lap_num + 1 if warning == "PIT_SOON" else None,
        compound_recommendation = "MEDIUM" if compound == "SOFT" else "HARD",
        stop_duration_p05       = 2.3,
        stop_duration_p50       = 2.8,
        stop_duration_p95       = 3.6,
        undercut_prob           = round(max(0.2, min(0.85, 0.5 + tlife_diff * 0.04 - gap_ahead * 0.08)), 3),
        undercut_target         = str(rival_row["Driver"]) if rival_row is not None else None,
        sc_reactive             = situation_out.sc_prob_3lap > CFG.sc_prob_threshold,
        reasoning               = (
            f"Pit P50 2.8s. Undercut prob {0.5 + (tlife_diff * 0.04 if rival_row is not None else 0):.2f}. "
            f"Recommended compound: {'MEDIUM' if compound == 'SOFT' else 'HARD'}."
        ),
    )


In [41]:
def run_replay_demo(target_laps: list[int]) -> list[dict]:
    """Replay the orchestrator over selected laps of NOR's Sakhir 2025 race.

    For each target lap:
    1. Retrieves NOR's full telemetry row from the parquet.
    2. Retrieves the rival directly ahead (Position = NOR.Position - 1) using
       timing-screen fields only — enforcing the single-driver data boundary.
    3. Computes gap_ahead_s and pace_delta_s from the timing data (as a real
       strategy engineer would from the timing screen).
    4. Builds realistic mock sub-agent outputs from the parquet values.
    5. Calls _run_mc_simulation + _build_orchestrator_prompt +
       structured_orchestrator_llm to produce a StrategyRecommendation.
    6. Prints a lap summary and returns all results.
    """
    results = []

    for lap_num in target_laps:
        row = _our[_our["LapNumber"] == lap_num]
        if row.empty:
            print(f"Lap {lap_num}: no data in parquet — skipping")
            continue
        row = row.iloc[0]

        # Rival directly ahead — timing-screen fields only
        our_pos    = int(row["Position"])
        ahead_pos  = our_pos - 1
        rival_candidates = _rivals[
            (_rivals["LapNumber"] == lap_num) & (_rivals["Position"] == ahead_pos)
        ]
        rival_row = rival_candidates.iloc[0] if not rival_candidates.empty else None

        # gap_ahead_s and pace_delta_s from timing screen
        gap_ahead  = 1.2  # fallback — in production from OpenF1 /v1/intervals
        pace_delta = 0.0
        if rival_row is not None:
            pace_delta = float(row["LapTime_s"]) - float(rival_row["LapTime_s"])
            # gap_ahead not in parquet; using a representative value per lap
            gap_ahead = max(0.3, 1.8 - abs(pace_delta) * 2.0)

        # Build mocks and RaceState
        pace_out, tire_out, situation_out, pit_out = _build_realistic_mocks(row, rival_row)
        race_state = _build_race_state(row, gap_ahead, pace_delta)

        # Routing
        active = _decide_agents_to_call(
            tire_warning  = tire_out.warning_level,
            sc_prob_3lap  = situation_out.sc_prob_3lap,
            radio_alerts  = [],
        )
        pit_out_active = pit_out if "N28" in active else None

        # MC simulation
        mc_results = _run_mc_simulation(
            pace_out      = pace_out,
            tire_out      = tire_out,
            situation_out = situation_out,
            pit_out       = pit_out_active,
            alpha         = race_state.risk_tolerance,
        )
        best_mc = max(mc_results, key=lambda s: mc_results[s]["score"])

        # LLM synthesis
        prompt = _build_orchestrator_prompt(
            race_state          = race_state,
            mc_results          = mc_results,
            best_mc             = best_mc,
            pace_reasoning      = pace_out.reasoning,
            tire_reasoning      = tire_out.reasoning,
            situation_reasoning = situation_out.reasoning,
            pit_reasoning       = pit_out_active.reasoning if pit_out_active else "",
            radio_reasoning     = "No radio alerts this lap.",
        )
        rec = structured_orchestrator_llm.invoke(prompt)
        rec.scenario_scores    = mc_results
        rec.regulation_context = ""

        # Print lap summary
        rival_str = f"{rival_row['Driver']} P{ahead_pos}" if rival_row is not None else "no rival data"
        print(f"\n{'─'*70}")
        print(f"  Lap {lap_num:>2}  |  P{our_pos}  |  {row['Compound']} TyreLife {int(row['TyreLife'])}  |  rival ahead: {rival_str}")
        print(f"  Tire   : {tire_out.warning_level}  cliff P10/P50/P90 = {tire_out.laps_to_cliff_p10}/{tire_out.laps_to_cliff_p50}/{tire_out.laps_to_cliff_p90}")
        print(f"  SC prob: {situation_out.sc_prob_3lap:.2f}  |  active agents: N25 N26 N27 N29 {' '.join(sorted(active))}")
        print(f"  MC best: {best_mc}  (score={mc_results[best_mc]['score']:+.3f})")
        print(f"  → ACTION: {rec.action}  (conf={rec.confidence:.2f})")
        print(f"  Reasoning: {rec.reasoning[:140]}...")

        results.append({"lap": lap_num, "action": rec.action,
                        "confidence": rec.confidence, "mc_best": best_mc,
                        "tire_warning": tire_out.warning_level, "rec": rec})

    return results


replay_results = run_replay_demo([8, 20, 46])


──────────────────────────────────────────────────────────────────────
  Lap  8  |  P3  |  SOFT TyreLife 11  |  rival ahead: RUS P2
  Tire   : PIT_SOON  cliff P10/P50/P90 = 1.7/2.4/3.3
  SC prob: 0.10  |  active agents: N25 N26 N27 N29 N28 N30
  MC best: OVERCUT  (score=+0.617)
  → ACTION: OVERCUT  (conf=0.85)
  Reasoning: The Monte Carlo simulation indicates that an Overcut has the highest expected position gain (+0.901), with a P10 and P90 score of +0.333 and...

──────────────────────────────────────────────────────────────────────
  Lap 20  |  P3  |  MEDIUM TyreLife 10  |  rival ahead: RUS P2
  Tire   : MONITOR  cliff P10/P50/P90 = 5.7/8.2/11.5
  SC prob: 0.11  |  active agents: N25 N26 N27 N29 
  MC best: OVERCUT  (score=+0.648)
  → ACTION: OVERCUT  (conf=0.85)
  Reasoning: The Monte Carlo simulation indicates that an Overcut has the highest expected position gain (+0.963), with a positive P10 and P90 score. Giv...

────────────────────────────────────────────────────────────────

### Step 5 results — End-to-end replay: NOR · Sakhir 2025

Three representative laps replayed with the full orchestrator pipeline. Sub-agents run as
realistic parquet-backed mocks; the MoE router, MC simulation, and LLM synthesis are all live.

| Lap | Compound | TyreLife | Position | Rival ahead | Tire warning | SC prob | Active agents | MC best | Action | Conf |
|-----|----------|----------|----------|-------------|--------------|---------|---------------|---------|--------|------|
| 8   | SOFT     | 11       | P3       | RUS P2      | PIT_SOON     | 0.10    | N25–N30 (all) | OVERCUT | OVERCUT | 0.85 |
| 20  | MEDIUM   | 10       | P3       | RUS P2      | MONITOR      | 0.11    | N25–N27 N29   | OVERCUT | OVERCUT | 0.85 |
| 46  | MEDIUM   | 14       | P4       | LEC P3      | PIT_SOON     | 0.23    | N25–N30 (all) | OVERCUT | OVERCUT | 0.95 |

**MoE routing behaves correctly.** N28 (Pit Strategy) and N30 (RAG) are activated only when
the tire agent reports `PIT_SOON` (laps 8 and 46). At lap 20 the tyre is in `MONITOR` state
with 5–8 laps to the cliff, so the router correctly skips them — no unnecessary computation.

**LLM recommendations are coherent.** All three laps recommend OVERCUT, which is the right
call for a driver in P3/P4 with a rival ahead who will pit first. The confidence rises from
0.85 to 0.95 at lap 46, where the combination of `PIT_SOON`, longer tyre life (14 laps),
and elevated SC probability (0.23) makes the case unambiguous.

**Single-driver perspective is enforced.** Rival data (RUS, LEC) is consumed from timing-screen
columns only (`Position`, `LapTime_s`, `Compound`, `TyreLife`, `SpeedST`). Internal rival
telemetry — degradation rate, fuel load, ERS — is never touched. The design constraint
documented in `documents/dev_docs/tasks/single_driver_perspective.md` holds naturally.

**What changes in v0.9.** The four mock functions (`_build_realistic_mocks`, `_build_race_state`,
`run_replay_demo`) will be replaced by real sub-agent calls (N25–N30 loaded in the same kernel)
and a proper `RaceStateManager` that handles the parquet→`RaceState` conversion with explicit
boundary enforcement. The orchestrator logic (MC simulation, LLM synthesis, routing) requires
no changes.

---

## Step 6 — Export orchestrator config

Exports `strategy_orchestrator_config_v1.json` to `data/models/agents/`.

The config captures everything needed to reconstruct the orchestrator in production:
routing thresholds, MC simulation parameters, agent list, LLM settings, and the
data boundary spec (which fields are available for our driver vs rivals).
This file is the handoff artifact for `src/agents/orchestrator.py` in v0.9.

In [42]:
EXPORT_DIR = repo_root / "data" / "models" / "agents"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

config = {
    "notebook": "N31_strategy_orchestrator",
    "version": "v1",
    "description": (
        "Strategy Orchestrator — end-to-end multi-agent supervisor. "
        "Integrates N25–N30 sub-agents through MoE routing, MC simulation, "
        "and LLM synthesis to produce per-lap StrategyRecommendations."
    ),

    # ── LLM ───────────────────────────────────────────────────────────────────
    "llm": {
        "model_name":  CFG.model_name,
        "base_url":    CFG.base_url,
        "temperature": CFG.temperature,
    },

    # ── MoE routing thresholds (global fallback) ──────────────────────────────
    "routing": {
        "sc_prob_threshold":       CFG.sc_prob_threshold,
        "tire_warning_pit_levels": ["PIT_SOON"],
        "always_active":           ["N25", "N26", "N27", "N29"],
        "conditional": {
            "N28": "tire_warning == PIT_SOON  OR  radio_alert IN [PIT_REQUEST, OVERTAKE_ALERT]",
            "N30": "tire_warning == PIT_SOON  OR  sc_prob_3lap >= sc_prob_threshold",
        },
    },

    # ── Circuit-cluster-aware thresholds (v0.9) ───────────────────────────────
    # Replaces the global sc_prob_threshold scalar in production.
    # Cluster assignments: data/processed/circuit_clustering/circuit_features_with_clusters_k4_2025.parquet
    # Validated against Pirelli compound nomination tiers (press.pirelli.com, 2023-2025)
    # and per-circuit SC probability values from Heilmeier et al. 2020
    # (Applied Sciences 10(12), 4229. DOI: 10.3390/app10124229).
    # See documents/dev_docs/tasks/circuit_cluster_thresholds.md for full rationale.
    "cluster_aware_thresholds": {
        "cluster_source": "data/processed/circuit_clustering/circuit_features_with_clusters_k4_2025.parquet",
        "academic_reference": "Heilmeier et al. 2020, DOI:10.3390/app10124229; Pirelli Motorsport bulletins 2023-2025",
        "sc_prob_threshold_by_cluster": {
            "0": 0.25,
            "1": 0.35,
            "2": 0.20,
            "3": 0.30,
        },
        "cluster_labels": {
            "0": "high_energy_deg",
            "1": "stable_technical",
            "2": "sc_haven_montreal",
            "3": "special_condition_outliers",
        },
        "n26_cliff_thresholds": {
            "pit_soon_by_cluster": {"0": 3, "1": 4, "2": 2, "3": 5},
            "monitor_by_cluster":  {"0": 7, "1": 8, "2": 5, "3": 9},
            "overrides_by_gp": {
                "Mexico City": {"pit_soon": 3, "monitor": 7},
            },
        },
    },

    # ── MC simulation ─────────────────────────────────────────────────────────
    "mc_simulation": {
        "n_simulations":          CFG.n_sim,
        "strategies":             ["STAY_OUT", "PIT_NOW", "UNDERCUT", "OVERCUT"],
        "risk_tolerance_default": CFG.risk_tolerance_default,
        "score_weights": {
            "position_gain": "alpha",
            "time_saved":    "1 - alpha",
            "sc_bonus":      0.3,
        },
    },

    # ── Sub-agents ────────────────────────────────────────────────────────────
    "sub_agents": {
        "N25": {"role": "Pace prediction",      "output": "PaceOutput",          "notebook": "N25_pace_agent"},
        "N26": {"role": "Tyre degradation",     "output": "TireOutput",          "notebook": "N26_tire_agent"},
        "N27": {"role": "Race situation",       "output": "RaceSituationOutput", "notebook": "N27_race_situation_agent"},
        "N28": {"role": "Pit strategy",         "output": "PitStrategyOutput",   "notebook": "N28_pit_strategy_agent"},
        "N29": {"role": "Radio / RCM parsing",  "output": "RadioOutput",         "notebook": "N29_radio_agent"},
        "N30": {"role": "RAG regulations",      "output": "str",                 "notebook": "N30_rag_agent"},
    },

    # ── Single-driver data boundary ───────────────────────────────────────────
    "data_boundary": {
        "description": (
            "System operates from the perspective of one driver on one team. "
            "Full telemetry for our driver; timing-screen data only for rivals. "
            "See documents/dev_docs/tasks/single_driver_perspective.md for full spec."
        ),
        "our_driver_fields": [
            "LapTime_s", "Sector1/2/3Time", "TyreLife", "Compound", "DegradationRate",
            "CumulativeDeg", "SpeedI1", "SpeedI2", "SpeedFL", "SpeedST",
            "Position", "Stint", "LapsSincePitStop", "FuelLoad",
            "AirTemp", "TrackTemp", "Humidity", "Rainfall",
        ],
        "rival_fields": [
            "Position", "LapTime_s", "Compound", "TyreLife", "SpeedST",
            "gap_ahead_s", "pace_delta_s", "tyre_life_diff",
        ],
        "rival_fields_NOT_available": [
            "DegradationRate", "FuelLoad", "ERS", "BrakeTemp",
            "TyrePressure", "EngineMode",
        ],
    },

    # ── v0.9 handoff notes ────────────────────────────────────────────────────
    "v09_handoff": {
        "src_target":    "src/agents/orchestrator.py",
        "state_manager": "src/agents/race_state_manager.py",
        "replace_mocks": (
            "_build_realistic_mocks → real run_pace_agent / run_tire_agent / ... calls. "
            "All sub-agents must be loaded in the same kernel session (or imported from src/)."
        ),
        "iter_laps": (
            "run_replay_demo loop → RaceStateManager.iter_laps(source) "
            "which handles parquet replay and OpenF1 live streaming."
        ),
        "cluster_thresholds": (
            "Replace CFG.sc_prob_threshold scalar with CFG.get_sc_threshold(circuit_cluster). "
            "cluster_id already available in SESSION_META in N26 and can be added to RaceState."
        ),
    },
}

out_path = EXPORT_DIR / "strategy_orchestrator_config_v1.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(f"Config exported → {out_path}")
print(f"Keys: {list(config.keys())}")

Config exported → c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\models\agents\strategy_orchestrator_config_v1.json
Keys: ['notebook', 'version', 'description', 'llm', 'routing', 'cluster_aware_thresholds', 'mc_simulation', 'sub_agents', 'data_boundary', 'v09_handoff']


### Step 6 results — Export

`strategy_orchestrator_config_v1.json` exported to `data/models/agents/`.
Includes `cluster_aware_thresholds` section with research-validated per-cluster values
(see `documents/dev_docs/tasks/circuit_cluster_thresholds.md`).

**N31 is complete.** The notebook covers the full orchestrator lifecycle:

| Step | What it builds |
|------|----------------|
| 0    | `OrchestratorCFG` — LLM client, thresholds, MC params |
| 1    | `RaceState` Pydantic model + `_decide_agents_to_call` MoE router |
| 2    | `_run_mc_simulation` — Monte Carlo over {STAY_OUT, PIT_NOW, UNDERCUT, OVERCUT} |
| 3    | `StrategyRecommendation` + `_build_orchestrator_prompt` + LLM structured output |
| 4    | `run_strategy_orchestrator()` — public entry point wiring all layers |
| 5    | End-to-end replay demo: NOR · Sakhir 2025, 3 laps, real parquet + live LLM |
| 6    | Config export → `data/models/agents/strategy_orchestrator_config_v1.json` |

**What is real vs mocked in the current notebook:**
- Real: MoE routing, Monte Carlo simulation, LLM call, `StrategyRecommendation` output
- Mocked: sub-agent calls (N25–N30 replaced by parquet-backed mocks with identical output shapes)
- Not real-time: replay iterates selected laps sequentially, no per-lap pacing

**Next step (v0.9):** extract to `src/agents/orchestrator.py` + `src/agents/race_state_manager.py`,
replace mocks with real sub-agent imports, and wire to FastAPI for the Streamlit interface.

---

### Future work — circuit-cluster-aware thresholds (v0.9 / v1.1)

The current orchestrator uses global scalar thresholds for all routing decisions. The v0.9
implementation will make these **cluster-aware**, following the same principle used inside
the ML models (N25 uses `circuit_cluster` as XGBoost feature; N27 uses it as LGBM CAT feature).

Cluster assignments from `data/processed/circuit_clustering/circuit_features_with_clusters_k4_2025.parquet`:

| Cluster | Circuits | Characteristics |
|---------|----------|-----------------|
| 0 — high-energy | Sakhir, Melbourne, Silverstone, Spa, Suzuka, Lusail, São Paulo, Zandvoort, Las Vegas | High deg (−0.24 to −0.96 s/lap), multi-stop, C1–C3 Pirelli tier |
| 1 — stable/technical | Monza, Budapest, Yas Island, Marina Bay, Jeddah, Baku, Imola, Barcelona, Austin, Miami, Spielberg, Shanghai | Low deg, 1-stop preferred, C2–C4 Pirelli tier |
| 2 — SC haven | Montréal (only) | +0.046 deg rate, avg 4.15 pit stops/race, Wall of Champions |
| 3 — outliers | Monaco, Mexico City | Monaco: 72-lap max stint, slow pit lane; Mexico: 780 hPa altitude |

**Threshold values — validated against Pirelli compound nomination tiers and Heilmeier et al. 2020
(Applied Sciences 10(12), 4229. DOI: 10.3390/app10124229):**

```python
# N31 OrchestratorCFG — replace scalar with dict in v0.9
SC_PROB_THRESHOLD_BY_CLUSTER = {
    0: 0.25,   # high-energy: higher base SC rate, activate N30 earlier
    1: 0.35,   # stable: lower SC rate, avoid false triggers
    2: 0.20,   # Montréal: SC dominates strategy, activate at lower probability
    3: 0.30,   # Monaco/Mexico: mixed — Mexico SC rate low, Monaco SC almost certain
}

# N26 TireAgentConfig — counter-intuitive but strategically sound:
# Cluster 0 (multi-stop): shorter warning because the team is already in reactive mode
# Cluster 1 (1-stop): longer warning because missing the single window is very costly
CLIFF_PIT_SOON_BY_CLUSTER = {0: 3, 1: 4, 2: 2, 3: 5}  # Montréal:2 (sudden graining); Monaco:5 (slow pit lane)
CLIFF_MONITOR_BY_CLUSTER  = {0: 7, 1: 8, 2: 5, 3: 9}
# Mexico City override: {pit_soon: 3, monitor: 7} — altitude reduces apparent deg but not strategic risk
```

**Only two agents need changes** — the rest are already circuit-aware via ML features:

| Agent | Circuit-awareness | Action needed |
|-------|-------------------|---------------|
| N25 Pace | `circuit_cluster` XGBoost feature ✓ | None |
| N26 Tire | `cluster_id` in SESSION_META but **not used for thresholds** | `get_cliff_thresholds(cluster, gp_name)` method in v0.9 |
| N27 Race Situation | N12 `circuit_cluster` CAT ✓; N14 `circuit_sc_rate` ✓ | None |
| N28 Pit Strategy | N16 `circuit_undercut_rate` ✓ | None |
| **N31 Orchestrator** | Global `sc_prob_threshold = 0.30` ✗ | `get_sc_threshold(circuit_cluster)` method in v0.9 |
| N29 Radio | No circuit dependency | None |
| N30 RAG | No circuit dependency | None |

Infrastructure cost is minimal: `cluster_id` already flows through `SESSION_META` in N26
and can be added to `RaceState` in N31 as a single integer field.